<a href="https://colab.research.google.com/github/Rishabh9559/medical-llama-3.2-3B-model/blob/main/demo_llm_runing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_PATH = "/content/drive/MyDrive/lora_model_16bit_mergedv2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072

In [ ]:
# Test a simple prompt
prompt = "What is Pulmonary fibrosis?"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

output_ids = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    do_sample=False
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

user

What is Pulmonary fibrosis?assistant

Pulmonary fibrosis is a chronic disease in which the lungs become scarred. The scar tissue makes it difficult for the lungs to expand and fill with air.


In [ ]:
import torch

# Test prompt
system_prompt = """
You are an expert medical and scientific AI assistant.

Your role is to provide clear, accurate, and well-organized medical information in a professional and educational tone.

Guidelines:
- Automatically determine the appropriate response length based on question complexity.
- Keep answers concise for simple questions and detailed for complex or multi-part questions.
- Use headings, bullet points, and step-by-step explanations where helpful.
- Explain medical terms in simple language while maintaining technical accuracy.
- Provide factual, evidence-based information only.
- Do NOT ask the user to specify word or token limits.
- Do NOT include conversational filler, apologies, or meta commentary.
- Do NOT mention policies, AI limitations, or safety disclaimers.
- Start directly with the answer.

Output Rules:
- Maintain logical flow.
- Avoid repetition.
"""

prompt = "what is appendectomy? in details"

# Apply chat template
inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Generate response
with torch.inference_mode():
    output_ids = model.generate(
        input_ids=inputs,
              # Hard cap only (LLM decides actual length)
        do_sample=False,         # Deterministic output
        temperature=0.2,         # Factual, stable responses
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode full output
decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\n===== FULL MODEL OUTPUT =====\n")
print(decoded)

# ---- Extract ONLY assistant response ----
if "assistant" in decoded:
    assistant_response = decoded.split("assistant", 1)[-1].strip()
else:
    assistant_response = decoded.strip()

print("\n===== ASSISTANT RESPONSE ONLY =====\n")
print(assistant_response)



===== FULL MODEL OUTPUT =====

system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024


You are an expert medical and scientific AI assistant.

Your role is to provide clear, accurate, and well-organized medical information in a professional and educational tone.

Guidelines:
- Automatically determine the appropriate response length based on question complexity.
- Keep answers concise for simple questions and detailed for complex or multi-part questions.
- Use headings, bullet points, and step-by-step explanations where helpful.
- Explain medical terms in simple language while maintaining technical accuracy.
- Provide factual, evidence-based information only.
- Do NOT ask the user to specify word or token limits.
- Do NOT include conversational filler, apologies, or meta commentary.
- Do NOT mention policies, AI limitations, or safety disclaimers.
- Start directly with the answer.

Output Rules:
- Maintain logical flow.
- Avoid repetition.
user

what is appendectomy? i